# Step 6–9 — ICU Deterioration Prediction (Improved V2 — FIXED)
This notebook is designed for the **Step 5** file `eicu_hourly_labeled_6h_12h_24h.csv`.

### Fix included
The previous notebook could raise `NameError: Xtr is not defined` because the sequence variables were only created inside another cell/loop.  
This version has a single, explicit sequence-building section that creates and stores all train/validation/test arrays **before** any model training starts.

### Steps
- **Step 6:** 24-hour temporal sequences + improved LSTM/GRU
- **Step 7:** Logistic Regression, Random Forest, XGBoost baselines
- **Step 8:** Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC and threshold analysis
- **Step 9:** Best-model ranking by F1/PR-AUC/Recall/ROC-AUC

Run **Runtime → Run all** from the top.


In [ ]:
# ============================================================
# 1. INSTALL / IMPORTS
# ============================================================
!pip -q install xgboost scikit-learn tensorflow

import os, gc, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras import layers, Model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Ready.")


In [ ]:
# ============================================================
# 2. LOAD STEP 5 CSV
# ============================================================
from google.colab import files

CSV_NAME = "eicu_hourly_labeled_6h_12h_24h.csv"

# If the file is not already in /content, upload it.
csv_path = f"/content/{CSV_NAME}"

if not os.path.exists(csv_path):
    print(f"Upload {CSV_NAME}")
    uploaded = files.upload()
    if CSV_NAME in uploaded:
        csv_path = f"/content/{CSV_NAME}"
    else:
        # accept the first CSV if its name differs
        csvs = [k for k in uploaded.keys() if k.lower().endswith(".csv")]
        if not csvs:
            raise FileNotFoundError("No CSV uploaded.")
        csv_path = f"/content/{csvs[0]}"

df = pd.read_csv(csv_path, low_memory=False)

print("Loaded:", csv_path)
print("Shape:", df.shape)
print("Columns:", len(df.columns))
display(df.head())


In [ ]:
# ============================================================
# 3. VERIFY REQUIRED COLUMNS AND BASIC DATA
# ============================================================
required = [
    "patientunitstayid", "hour",
    "label_6h", "label_12h", "label_24h",
    "split"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required Step 5 columns: {missing}")

df["patientunitstayid"] = pd.to_numeric(
    df["patientunitstayid"], errors="coerce"
).astype("Int64")

df["hour"] = pd.to_numeric(df["hour"], errors="coerce")
for h in [6, 12, 24]:
    df[f"label_{h}h"] = pd.to_numeric(
        df[f"label_{h}h"], errors="coerce"
    ).fillna(0).astype(int)

df["split"] = df["split"].astype(str).str.lower().str.strip()

print("Split counts:")
display(df["split"].value_counts())

print("\nPositive label counts:")
for h in [6, 12, 24]:
    print(f"{h}h:", int(df[f"label_{h}h"].sum()))

print("\nRequired columns verified.")


In [ ]:
# ============================================================
# 4. SELECT SAFE CLINICAL FEATURES
# ============================================================
# Do NOT use labels, discharge/event timing, IDs, or obvious
# outcome-derived fields as model inputs.

EXCLUDE_EXACT = {
    "patientunitstayid", "hour",
    "label_6h", "label_12h", "label_24h",
    "split",
    "icu_death", "hours_to_event",
    "discharge_hour", "unitdischargeoffset",
    "unitdischargestatus", "actualdeath", "death"
}

EXCLUDE_WORDS = [
    "mortality", "mortality_", "death", "deceased",
    "discharge", "discharged", "outcome",
    "hours_to_event", "event_time", "apache"
]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

feature_cols = []
for c in numeric_cols:
    cl = c.lower()
    if c in EXCLUDE_EXACT:
        continue
    if any(w in cl for w in EXCLUDE_WORDS):
        continue
    feature_cols.append(c)

# Remove columns that are completely missing
feature_cols = [
    c for c in feature_cols
    if df[c].notna().any()
]

if len(feature_cols) == 0:
    raise ValueError("No usable numeric clinical features were found.")

print("Number of features:", len(feature_cols))
print(feature_cols)


In [ ]:
# ============================================================
# 5. SORT DATA + CLEAN SPLITS
# ============================================================
df = df.dropna(subset=["patientunitstayid", "hour"]).copy()
df["patientunitstayid"] = df["patientunitstayid"].astype(int)

df = df.sort_values(
    ["patientunitstayid", "hour"]
).reset_index(drop=True)

valid_splits = {"train", "val", "validation", "test"}
bad = set(df["split"].unique()) - valid_splits
if bad:
    print("Warning: unexpected split values:", bad)

df.loc[df["split"].eq("validation"), "split"] = "val"

print(df["split"].value_counts())


## Step 6A — Training-only imputation and scaling

The imputer and scaler are fitted **only on training rows**.  
Validation and test rows are transformed using those training parameters.


In [ ]:
# ============================================================
# 6. TRAIN-ONLY IMPUTATION + SCALING
# ============================================================
train_mask = df["split"].eq("train")
val_mask   = df["split"].eq("val")
test_mask  = df["split"].eq("test")

if train_mask.sum() == 0 or val_mask.sum() == 0 or test_mask.sum() == 0:
    raise ValueError("Train/val/test split is required in the Step 5 file.")

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

train_values = imputer.fit_transform(df.loc[train_mask, feature_cols])
imputer.fit(df.loc[train_mask, feature_cols])

# Fit scaler on imputed training data only
train_imputed = imputer.transform(df.loc[train_mask, feature_cols])
scaler.fit(train_imputed)

all_imputed = imputer.transform(df[feature_cols])
all_scaled = scaler.transform(all_imputed).astype(np.float32)

df_scaled = df[["patientunitstayid", "hour", "split",
                "label_6h", "label_12h", "label_24h"]].copy()

X_scaled_df = pd.DataFrame(
    all_scaled,
    columns=feature_cols,
    index=df.index
)

print("Scaled matrix:", all_scaled.shape)


## Step 6B — Robust 24-hour sequence generation

A sequence contains **24 consecutive hourly observations** from the same ICU stay.

The function below explicitly returns:

`X_train, y_train, X_val, y_val, X_test, y_test`

and stores them in `seq_cache`.

This is the part that fixes the previous `Xtr is not defined` problem.


In [ ]:
# ============================================================
# 7. EXPLICIT SEQUENCE GENERATION — NO Xtr NameError
# ============================================================
INPUT_HOURS = 24

def build_sequences(
    original_df,
    scaled_values,
    feature_columns,
    horizon,
    input_hours=24
):
    X_parts = {"train": [], "val": [], "test": []}
    y_parts = {"train": [], "val": [], "test": []}

    # Work by patient so no sequence crosses ICU stays.
    for pid, idx in original_df.groupby(
        "patientunitstayid", sort=False
    ).groups.items():

        idx = np.asarray(idx)
        idx = idx[np.argsort(
            original_df.loc[idx, "hour"].to_numpy()
        )]

        hours = original_df.loc[idx, "hour"].to_numpy()
        splits = original_df.loc[idx, "split"].to_numpy()
        labels = original_df.loc[
            idx, f"label_{horizon}h"
        ].to_numpy(dtype=np.int8)

        # Consecutive-hour check.
        for end in range(input_hours - 1, len(idx)):
            start = end - input_hours + 1

            window_hours = hours[start:end + 1]

            if not np.all(np.diff(window_hours) == 1):
                continue

            # Keep the sequence in one split only.
            split_values = splits[start:end + 1]
            split_name = split_values[-1]

            if split_name not in X_parts:
                continue

            if not np.all(split_values == split_name):
                continue

            seq_idx = idx[start:end + 1]

            X_seq = scaled_values[seq_idx]
            y_value = int(labels[end])

            X_parts[split_name].append(X_seq)
            y_parts[split_name].append(y_value)

    out = {}

    for s in ["train", "val", "test"]:
        if not X_parts[s]:
            raise ValueError(
                f"No sequences generated for {s} at {horizon}h."
            )

        out[s] = (
            np.stack(X_parts[s]).astype(np.float32),
            np.asarray(y_parts[s], dtype=np.int32)
        )

    return out


seq_cache = {}

for h in [6, 12, 24]:
    seq_cache[h] = build_sequences(
        df,
        all_scaled,
        feature_cols,
        horizon=h,
        input_hours=INPUT_HOURS
    )

    X_train, y_train = seq_cache[h]["train"]
    X_val, y_val = seq_cache[h]["val"]
    X_test, y_test = seq_cache[h]["test"]

    print(f"\n{h}h")
    print("X_train:", X_train.shape, "positives:", int(y_train.sum()))
    print("X_val:  ", X_val.shape,   "positives:", int(y_val.sum()))
    print("X_test: ", X_test.shape,  "positives:", int(y_test.sum()))

print("\nSequence generation completed successfully.")


In [ ]:
# ============================================================
# 8. FINAL EXPLICIT VARIABLE CHECK
# ============================================================
# These variables are deliberately created here before training.
# This guarantees that a training cell cannot depend on an
# accidentally skipped sequence-generation cell.

Xtr, ytr = seq_cache[6]["train"]
Xv,  yv  = seq_cache[6]["val"]
Xte, yte = seq_cache[6]["test"]

assert Xtr.ndim == 3
assert Xv.ndim == 3
assert Xte.ndim == 3
assert Xtr.shape[2] == len(feature_cols)

print("Xtr:", Xtr.shape)
print("Xv :", Xv.shape)
print("Xte:", Xte.shape)
print("Feature count:", len(feature_cols))
print("Sequence variables are READY.")


## Step 6C — Improved LSTM/GRU

The models use:
- 24-hour temporal input
- recurrent layer
- Layer Normalization
- Dropout
- Attention
- Dense classifier

The loss uses focal loss to give more learning emphasis to difficult/rare positive cases.


In [ ]:
# ============================================================
# 9. MODEL + FOCAL LOSS
# ============================================================
def binary_focal_loss(gamma=2.0, alpha=0.75):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(
            y_pred, tf.keras.backend.epsilon(),
            1.0 - tf.keras.backend.epsilon()
        )

        pt = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)

        return tf.reduce_mean(
            -alpha_t * tf.pow(1.0 - pt, gamma) * tf.math.log(pt)
        )
    return loss


def build_improved_model(input_shape, kind="LSTM"):
    inputs = layers.Input(shape=input_shape)

    if kind.upper() == "GRU":
        x = layers.GRU(
            64,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.05
        )(inputs)
        x = layers.GRU(
            32,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.05
        )(x)
    else:
        x = layers.LSTM(
            64,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.05
        )(inputs)
        x = layers.LSTM(
            32,
            return_sequences=True,
            dropout=0.20,
            recurrent_dropout=0.05
        )(x)

    x = layers.LayerNormalization()(x)

    # Lightweight self-attention
    attn = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=16,
        dropout=0.10
    )(x, x)

    x = layers.Add()([x, attn])
    x = layers.LayerNormalization()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.30)(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=3e-4
        ),
        loss=binary_focal_loss(
            gamma=2.0,
            alpha=0.75
        ),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(
                name="roc_auc",
                curve="ROC"
            ),
            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR"
            )
        ]
    )

    return model


def threshold_by_val_f1(y_true, prob):
    thresholds = np.linspace(0.01, 0.99, 197)

    best_t = 0.50
    best_f1 = -1.0

    for t in thresholds:
        pred = (prob >= t).astype(int)
        score = f1_score(
            y_true,
            pred,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_t = float(t)

    return best_t, best_f1


def evaluate_predictions(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)

    result = {
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(
            y_true, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, pred, zero_division=0
        ),
        "ROC-AUC": np.nan,
        "PR-AUC": np.nan,
        "Threshold": threshold
    }

    if len(np.unique(y_true)) == 2:
        result["ROC-AUC"] = roc_auc_score(
            y_true, prob
        )
        result["PR-AUC"] = average_precision_score(
            y_true, prob
        )

    return result


## Step 6D — Train LSTM and GRU

The loop below defines `Xtr`, `Xv`, and `Xte` **inside the same training cell** before `build_improved_model()` is called.


In [ ]:
# ============================================================
# 10. TRAIN LSTM + GRU FOR 6h / 12h / 24h
# ============================================================
EPOCHS = 40
BATCH_SIZE = 128

os.makedirs(
    "/content/step6_9_v2_fixed_outputs",
    exist_ok=True
)

results = []
predictions = {}
histories = {}

for h in [6, 12, 24]:

    # CRITICAL: create variables BEFORE model construction.
    Xtr, ytr = seq_cache[h]["train"]
    Xv,  yv  = seq_cache[h]["val"]
    Xte, yte = seq_cache[h]["test"]

    print("\n" + "=" * 70)
    print(f"HORIZON: {h}h")
    print("Train:", Xtr.shape, "positive:", int(ytr.sum()))
    print("Val:  ", Xv.shape,  "positive:", int(yv.sum()))
    print("Test: ", Xte.shape, "positive:", int(yte.sum()))
    print("=" * 70)

    # Moderate positive weighting. Focal loss already handles
    # imbalance, so avoid an extreme class weight.
    counts = np.bincount(ytr, minlength=2)
    pos = max(1, counts[1])
    neg = max(1, counts[0])

    ratio = neg / pos
    pos_weight = float(
        np.clip(np.sqrt(ratio), 1.0, 6.0)
    )

    class_weight = {
        0: 1.0,
        1: pos_weight
    }

    print("Positive class weight:", round(pos_weight, 3))

    for kind in ["LSTM", "GRU"]:

        print(f"\n---------- {kind} {h}h ----------")

        # Xtr is guaranteed to exist here.
        model = build_improved_model(
            input_shape=Xtr.shape[1:],
            kind=kind
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_pr_auc",
                mode="max",
                patience=7,
                restore_best_weights=True,
                verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_pr_auc",
                mode="max",
                factor=0.5,
                patience=3,
                min_lr=1e-6,
                verbose=1
            ),
            tf.keras.callbacks.ModelCheckpoint(
                f"/content/step6_9_v2_fixed_outputs/"
                f"{kind.lower()}_{h}h_best.keras",
                monitor="val_pr_auc",
                mode="max",
                save_best_only=True,
                verbose=0
            )
        ]

        history = model.fit(
            Xtr,
            ytr,
            validation_data=(Xv, yv),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            class_weight=class_weight,
            callbacks=callbacks,
            verbose=1
        )

        histories[f"{kind}_{h}h"] = history.history

        p_val = model.predict(
            Xv,
            batch_size=BATCH_SIZE,
            verbose=0
        ).ravel()

        p_test = model.predict(
            Xte,
            batch_size=BATCH_SIZE,
            verbose=0
        ).ravel()

        threshold, val_f1 = threshold_by_val_f1(
            yv,
            p_val
        )

        metrics = evaluate_predictions(
            yte,
            p_test,
            threshold
        )

        metrics.update({
            "Horizon": f"{h}h",
            "Model": kind,
            "Val_F1": val_f1
        })

        results.append(metrics)

        predictions[f"{kind}_{h}h"] = {
            "y_true": yte.copy(),
            "prob": p_test.copy(),
            "threshold": threshold
        }

        model.save(
            f"/content/step6_9_v2_fixed_outputs/"
            f"{kind.lower()}_{h}h_final.keras"
        )

        print("Threshold:", round(threshold, 4))
        print("Val F1:", round(val_f1, 4))
        print("Test:")
        for k in [
            "Accuracy", "Precision", "Recall",
            "F1", "ROC-AUC", "PR-AUC"
        ]:
            print(f"  {k}: {metrics[k]:.4f}")

        del model
        tf.keras.backend.clear_session()
        gc.collect()


deep_results = pd.DataFrame(results)

deep_results = deep_results[
    [
        "Horizon", "Model",
        "Accuracy", "Precision", "Recall",
        "F1", "ROC-AUC", "PR-AUC",
        "Threshold", "Val_F1"
    ]
].sort_values(
    ["Horizon", "F1"],
    ascending=[True, False]
).reset_index(drop=True)

print("\nDEEP LEARNING RESULTS")
display(deep_results.round(4))

deep_results.to_csv(
    "/content/step6_9_v2_fixed_outputs/"
    "deep_learning_results.csv",
    index=False
)


## Step 7 — Classical ML baselines

For the non-sequential models, each 24-hour sequence is converted into summary features:
- last value
- mean
- standard deviation
- linear trend

This preserves some temporal information while making the input compatible with classical ML.


In [ ]:
# ============================================================
# 11. CONVERT 24h SEQUENCES TO ML SUMMARY FEATURES
# ============================================================
def sequence_to_summary(X):
    last = X[:, -1, :]
    mean = np.mean(X, axis=1)
    std = np.std(X, axis=1)

    # Linear trend across 24 time points
    t = np.arange(X.shape[1], dtype=np.float32)
    t = t - t.mean()
    denom = np.sum(t ** 2)

    centered = X - np.mean(X, axis=1, keepdims=True)
    trend = np.sum(
        centered * t.reshape(1, -1, 1),
        axis=1
    ) / denom

    return np.concatenate(
        [last, mean, std, trend],
        axis=1
    ).astype(np.float32)


ml_cache = {}

for h in [6, 12, 24]:

    Xtr, ytr = seq_cache[h]["train"]
    Xv, yv = seq_cache[h]["val"]
    Xte, yte = seq_cache[h]["test"]

    ml_cache[h] = {
        "train": (sequence_to_summary(Xtr), ytr),
        "val":   (sequence_to_summary(Xv), yv),
        "test":  (sequence_to_summary(Xte), yte)
    }

    print(
        h,
        ml_cache[h]["train"][0].shape,
        ml_cache[h]["val"][0].shape,
        ml_cache[h]["test"][0].shape
    )


In [ ]:
# ============================================================
# 12. TRAIN LOGISTIC REGRESSION / RANDOM FOREST / XGBOOST
# ============================================================
ml_results = []

for h in [6, 12, 24]:

    Xm_tr, ym_tr = ml_cache[h]["train"]
    Xm_v,  ym_v  = ml_cache[h]["val"]
    Xm_te, ym_te = ml_cache[h]["test"]

    print("\n" + "=" * 70)
    print(f"CLASSICAL ML — {h}h")
    print("=" * 70)

    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            C=0.5,
            solver="liblinear",
            random_state=SEED
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=3,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=SEED
        ),

        "XGBoost": XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            reg_lambda=2.0,
            objective="binary:logistic",
            eval_metric="aucpr",
            tree_method="hist",
            n_jobs=-1,
            random_state=SEED
        )
    }

    for name, clf in models.items():

        print("Training:", name)

        clf.fit(Xm_tr, ym_tr)

        pv = clf.predict_proba(Xm_v)[:, 1]
        pte = clf.predict_proba(Xm_te)[:, 1]

        threshold, val_f1 = threshold_by_val_f1(
            ym_v, pv
        )

        metrics = evaluate_predictions(
            ym_te,
            pte,
            threshold
        )

        metrics.update({
            "Horizon": f"{h}h",
            "Model": name,
            "Val_F1": val_f1
        })

        ml_results.append(metrics)

        print(
            name,
            "| F1:", round(metrics["F1"], 4),
            "| Recall:", round(metrics["Recall"], 4),
            "| PR-AUC:", round(metrics["PR-AUC"], 4)
        )

        del clf
        gc.collect()


ml_results_df = pd.DataFrame(ml_results)

ml_results_df = ml_results_df[
    [
        "Horizon", "Model",
        "Accuracy", "Precision", "Recall",
        "F1", "ROC-AUC", "PR-AUC",
        "Threshold", "Val_F1"
    ]
]

display(
    ml_results_df
    .sort_values(["Horizon", "F1"], ascending=[True, False])
    .round(4)
)

ml_results_df.to_csv(
    "/content/step6_9_v2_fixed_outputs/"
    "classical_ml_results.csv",
    index=False
)


## Step 8 — Final evaluation and comparison


In [ ]:
# ============================================================
# 13. COMBINE ALL MODEL RESULTS
# ============================================================
all_results = pd.concat(
    [deep_results, ml_results_df],
    ignore_index=True
)

all_results = all_results[
    [
        "Horizon", "Model",
        "Accuracy", "Precision", "Recall",
        "F1", "ROC-AUC", "PR-AUC",
        "Threshold", "Val_F1"
    ]
]

all_results["F1"] = all_results["F1"].astype(float)
all_results["PR-AUC"] = all_results["PR-AUC"].astype(float)
all_results["Recall"] = all_results["Recall"].astype(float)
all_results["ROC-AUC"] = all_results["ROC-AUC"].astype(float)

all_results = all_results.sort_values(
    ["Horizon", "F1", "PR-AUC"],
    ascending=[True, False, False]
).reset_index(drop=True)

print("STEP 8 — FINAL MODEL COMPARISON")
display(all_results.round(4))

all_results.to_csv(
    "/content/step6_9_v2_fixed_outputs/"
    "STEP8_FINAL_MODEL_COMPARISON.csv",
    index=False
)


In [ ]:
# ============================================================
# 14. CONFUSION MATRICES FOR ALL MODELS
# ============================================================
for h in ["6h", "12h", "24h"]:

    subset = all_results[
        all_results["Horizon"] == h
    ].sort_values("F1", ascending=False)

    print("\n" + "=" * 70)
    print("HORIZON:", h)
    print("=" * 70)

    display(subset.round(4))

    # Deep-learning predictions are available in memory.
    for _, row in subset.iterrows():

        key = f"{row['Model']}_{h}"

        if key not in predictions:
            continue

        item = predictions[key]

        y_true = item["y_true"]
        prob = item["prob"]
        threshold = item["threshold"]

        pred = (prob >= threshold).astype(int)

        cm = confusion_matrix(y_true, pred)

        print(
            f"{row['Model']} — threshold={threshold:.3f}"
        )

        plt.figure(figsize=(4, 3))
        plt.imshow(cm)
        plt.title(f"{row['Model']} — {h}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.xticks([0, 1], ["No", "Yes"])
        plt.yticks([0, 1], ["No", "Yes"])

        for i in range(2):
            for j in range(2):
                plt.text(
                    j, i,
                    str(cm[i, j]),
                    ha="center",
                    va="center"
                )

        plt.tight_layout()
        plt.show()


In [ ]:
# ============================================================
# 15. ROC + PRECISION-RECALL CURVES FOR DEEP MODELS
# ============================================================
for h in [6, 12, 24]:

    plt.figure(figsize=(7, 5))

    for kind in ["LSTM", "GRU"]:

        key = f"{kind}_{h}h"

        if key not in predictions:
            continue

        item = predictions[key]
        y_true = item["y_true"]
        prob = item["prob"]

        if len(np.unique(y_true)) < 2:
            continue

        fpr, tpr, _ = roc_curve(
            y_true, prob
        )

        auc = roc_auc_score(
            y_true, prob
        )

        plt.plot(
            fpr, tpr,
            label=f"{kind} AUC={auc:.3f}"
        )

    plt.plot(
        [0, 1], [0, 1],
        linestyle="--"
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {h}h")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    plt.figure(figsize=(7, 5))

    for kind in ["LSTM", "GRU"]:

        key = f"{kind}_{h}h"

        if key not in predictions:
            continue

        item = predictions[key]
        y_true = item["y_true"]
        prob = item["prob"]

        if len(np.unique(y_true)) < 2:
            continue

        precision, recall, _ = precision_recall_curve(
            y_true, prob
        )

        ap = average_precision_score(
            y_true, prob
        )

        plt.plot(
            recall, precision,
            label=f"{kind} AP={ap:.3f}"
        )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision–Recall Curve — {h}h")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


## Step 9 — Best model selection

For this imbalanced deterioration task, accuracy is **not** used as the primary ranking criterion.

The ranking emphasizes:
1. F1
2. PR-AUC
3. Recall
4. ROC-AUC


In [ ]:
# ============================================================
# 16. STEP 9 — RANK MODELS
# ============================================================
ranking = all_results.copy()

# Normalize ranks within each horizon.
ranking["Rank_F1"] = ranking.groupby(
    "Horizon"
)["F1"].rank(
    ascending=False,
    method="min"
)

ranking["Rank_PR"] = ranking.groupby(
    "Horizon"
)["PR-AUC"].rank(
    ascending=False,
    method="min"
)

ranking["Rank_Recall"] = ranking.groupby(
    "Horizon"
)["Recall"].rank(
    ascending=False,
    method="min"
)

ranking["Rank_ROC"] = ranking.groupby(
    "Horizon"
)["ROC-AUC"].rank(
    ascending=False,
    method="min"
)

ranking["Overall_Rank_Score"] = (
    0.35 * ranking["Rank_F1"] +
    0.30 * ranking["Rank_PR"] +
    0.20 * ranking["Rank_Recall"] +
    0.15 * ranking["Rank_ROC"]
)

ranking = ranking.sort_values(
    ["Horizon", "Overall_Rank_Score"]
).reset_index(drop=True)

print("STEP 9 — MODEL RANKING")
display(
    ranking[
        [
            "Horizon", "Model",
            "F1", "PR-AUC", "Recall",
            "ROC-AUC", "Overall_Rank_Score"
        ]
    ].round(4)
)

ranking.to_csv(
    "/content/step6_9_v2_fixed_outputs/"
    "STEP9_MODEL_RANKING.csv",
    index=False
)

best_by_horizon = (
    ranking
    .groupby("Horizon", as_index=False)
    .first()
)

print("\nBEST MODEL BY HORIZON")
display(
    best_by_horizon[
        [
            "Horizon", "Model",
            "F1", "PR-AUC", "Recall",
            "ROC-AUC", "Overall_Rank_Score"
        ]
    ].round(4)
)

best_by_horizon.to_csv(
    "/content/step6_9_v2_fixed_outputs/"
    "STEP9_BEST_MODEL_BY_HORIZON.csv",
    index=False
)


In [ ]:
# ============================================================
# 17. PROJECT SUMMARY
# ============================================================
print("=" * 75)
print("ICU DETERIORATION PREDICTION — STEP 6–9 SUMMARY")
print("=" * 75)

for _, r in best_by_horizon.iterrows():
    print(
        f"{r['Horizon']}: {r['Model']} | "
        f"F1={r['F1']:.3f} | "
        f"Recall={r['Recall']:.3f} | "
        f"PR-AUC={r['PR-AUC']:.3f} | "
        f"ROC-AUC={r['ROC-AUC']:.3f}"
    )

print("\nFiles saved in:")
print("/content/step6_9_v2_fixed_outputs/")

print("\nMain files:")
for f in sorted(
    os.listdir("/content/step6_9_v2_fixed_outputs")
):
    print(" -", f)


In [ ]:
# ============================================================
# 18. OPTIONAL: ZIP ALL STEP 6–9 OUTPUTS
# ============================================================
import shutil

output_dir = "/content/step6_9_v2_fixed_outputs"
zip_base = "/content/step6_9_v2_fixed_outputs"

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    output_dir
)

print("Created:", zip_path)
print("You can download it from the Colab Files panel.")
